# ⚽ Phase 2b — Revised Build & Train (two fixes)

v1 had a clear problem: the model shouted **"draw"** at almost everything. It caught draws (recall 0.825) only by guessing draw ~9,800 times when there were just 3,501 real draws. Like a smoke alarm that screams at burnt toast — technically never misses a fire, but useless.

**Two fixes in this version:**
```
FIX 1  Gentler class weights (sqrt of inverse-frequency, not full inverse)
       -> nudges the model toward draws instead of shoving it
FIX 2  Slightly bigger net: 24->12  becomes  40->20
       -> more room to learn the real pattern instead of a lazy shortcut
```
Everything else stays the same: same data, same warm-start, same two-stage training, same 3-class (win/draw/lose) setup. We keep the draw class — we just stop the model abusing it.

At the end this notebook prints a **side-by-side v1 vs v2** so we can see if the middle column of the confusion matrix actually thinned out.


## Cell 1 — Setup (same as before)

In [10]:
import json, time, pickle, warnings
import numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report, precision_score, recall_score

ROOT      = Path('.')
DATA_PROC = ROOT / 'data' / 'processed'
MODELS    = ROOT / 'models'
RESULTS   = ROOT / 'results'
for d in (MODELS, RESULTS): d.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)
print('Device:', DEVICE)


Device: cpu


## Cell 2 — Load data & split (identical to v1 so the comparison is fair)

In [11]:
FEATURE_COLS = ['goal_diff','minute_norm','is_second_half','home_rank_norm',
                'away_rank_norm','rank_diff','is_knockout','lead_changes_norm',
                'is_neutral_venue','score_state','strength_x_time']
TARGET_COL = 'outcome'
N_FEATURES, N_CLASSES = len(FEATURE_COLS), 3

df = pd.read_parquet(DATA_PROC / 'features_v2.parquet')
X = df[FEATURE_COLS].values.astype('float32')
y = df[TARGET_COL].values.astype('int64')
groups = df['match_id'].values

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trainval_idx, test_idx = next(gss1.split(X, y, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_rel, va_rel = next(gss2.split(X[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx = trainval_idx[tr_rel]; val_idx = trainval_idx[va_rel]

scaler = StandardScaler().fit(X[train_idx])
Xtr = scaler.transform(X[train_idx]).astype('float32')
Xva = scaler.transform(X[val_idx]).astype('float32')
Xte = scaler.transform(X[test_idx]).astype('float32')
ytr, yva, yte = y[train_idx], y[val_idx], y[test_idx]

def loader(Xa, ya, bs=256, shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(Xa), torch.tensor(ya)), batch_size=bs, shuffle=shuffle)
train_loader = loader(Xtr, ytr, shuffle=True); val_loader = loader(Xva, yva)
print(f'train {len(train_idx):,} | val {len(val_idx):,} | test {len(test_idx):,} snapshots (same split as v1)')


train 44,785 | val 14,901 | test 14,912 snapshots (same split as v1)


## Cell 3 — FIX 1: gentler class weights

In v1 we weighted each class by **1 / frequency** (full inverse). That over-punished the model for missing draws, so it spammed draws.

Now we use **1 / sqrt(frequency)** — the square-root version. Why does a square root make it gentler? Because square root pulls big numbers down toward small ones. Worked example, every step, using the real training frequencies (away 31.7%, draw 26.4%, home 41.9%):

Full inverse (v1 style):
- away: 1 / 0.317 = 3.155
- draw: 1 / 0.264 = 3.788   <- draw is 3.788/2.387 = 1.59x the home weight
- home: 1 / 0.419 = 2.387

Square-root inverse (v2 style) — step by step for draw:
- step 1: take the frequency, 0.264
- step 2: square-root it: sqrt(0.264) = 0.514
- step 3: invert: 1 / 0.514 = 1.946
- do the same for the others: away 1/sqrt(0.317)=1.776, home 1/sqrt(0.419)=1.545

Now draw is only 1.946/1.545 = 1.26x the home weight, instead of 1.59x. So the draw still gets *some* extra help (it's still the biggest weight), but the shove became a nudge. That's the whole fix: keep the direction, shrink the force.


In [12]:
counts = np.bincount(ytr, minlength=3).astype('float64')
freqs  = counts / counts.sum()

# v1 style (full inverse) — shown only for comparison
w_full = 1.0 / freqs
w_full = w_full / w_full.sum() * 3.0

# v2 style (sqrt inverse) — the gentler version we actually use
w_sqrt = 1.0 / np.sqrt(freqs)
w_sqrt = w_sqrt / w_sqrt.sum() * 3.0

class_weights = torch.tensor(w_sqrt, dtype=torch.float32, device=DEVICE)
print('Class          away    draw    home')
print('Frequency     ', ' '.join(f'{f:6.1%}' for f in freqs))
print('v1 weight     ', ' '.join(f'{w:6.2f}' for w in w_full), ' (full inverse - too harsh)')
print('v2 weight     ', ' '.join(f'{w:6.2f}' for w in w_sqrt), ' (sqrt inverse - gentle nudge)')
print('\n-> draw is still the biggest weight, but closer to the others now.')


Class          away    draw    home
Frequency       31.7%  26.4%  41.9%
v1 weight        1.02   1.22   0.77  (full inverse - too harsh)
v2 weight        1.01   1.11   0.88  (sqrt inverse - gentle nudge)

-> draw is still the biggest weight, but closer to the others now.


## Cell 4 — FIX 2: a slightly bigger brain

v1 was 11 -> 24 -> 12 -> 3, which is 627 learnable numbers. That flat 0.469 accuracy smelled like **underfitting** — the brain was so small it couldn't capture the real pattern, so it fell back on the dumb "guess draw" trick.

v2 is 11 -> 40 -> 20 -> 3. Let me count the parameters so you see it's still small, just less cramped:
- fc1: (11 inputs x 40 neurons) + 40 biases = 440 + 40 = 480
- fc2: (40 x 20) + 20 = 800 + 20 = 820
- head: (20 x 3) + 3 = 60 + 3 = 63
- total = 480 + 820 + 63 = 1,363 numbers

So we roughly doubled capacity (627 -> 1,363) — enough extra room to learn, but nowhere near big enough to memorise 45,000 training snapshots. Still small, just not starved.

We keep the same warm-start: donate the overlapping slice of the NBA first layer.


In [13]:
class FootballNet(nn.Module):
    def __init__(self, n_features=11, n_classes=3, h1=40, h2=20, dropout=0.30):
        super().__init__()
        self.fc1  = nn.Linear(n_features, h1)
        self.fc2  = nn.Linear(h1, h2)
        self.head = nn.Linear(h2, n_classes)
        self.drop = nn.Dropout(dropout); self.act = nn.ReLU()
    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.act(self.fc2(x)))
        return self.head(x)

model = FootballNet().to(DEVICE)
print(model)
print('Total params:', sum(p.numel() for p in model.parameters()))

# Warm-start from NBA
nba_candidates = [MODELS/'nba_base.pth',
                  Path('../nba_win_prob/model/win_prob_net.pth'),
                  Path('nba_win_prob/model/win_prob_net.pth')]
nba_path = next((p for p in nba_candidates if p.exists()), None)
TRANSFER_DONE = False
if nba_path:
    nba_ckpt = torch.load(nba_path, map_location='cpu', weights_only=False)
    nba_state = nba_ckpt.get('model_state_dict', nba_ckpt)
    nba_first = None
    for k,v in nba_state.items():
        if v.ndim==2 and v.shape[1] >= 7: nba_first=v; break
    if nba_first is not None:
        r = min(model.fc1.weight.shape[0], nba_first.shape[0])
        c = min(model.fc1.weight.shape[1], nba_first.shape[1])
        with torch.no_grad():
            model.fc1.weight.data[:r,:c] = nba_first[:r,:c].float()
        print(f'Warm-started: donated {r}x{c} slice from NBA layer (shape {tuple(nba_first.shape)})')
        TRANSFER_DONE = True
else:
    print('NBA weights not found — training from scratch (still works).')


FootballNet(
  (fc1): Linear(in_features=11, out_features=40, bias=True)
  (fc2): Linear(in_features=40, out_features=20, bias=True)
  (head): Linear(in_features=20, out_features=3, bias=True)
  (drop): Dropout(p=0.3, inplace=False)
  (act): ReLU()
)
Total params: 1363
NBA weights not found — training from scratch (still works).


## Cell 5 — Two-stage training (same recipe as v1)

In [14]:
def run_epoch(net, loader, optimizer=None):
    train_mode = optimizer is not None
    net.train() if train_mode else net.eval()
    crit = nn.CrossEntropyLoss(weight=class_weights)
    total, LG, LY = 0.0, [], []
    with torch.set_grad_enabled(train_mode):
        for xb,yb in loader:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = net(xb); loss = crit(logits, yb)
            if train_mode: optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item()*len(xb); LG.append(logits.detach().cpu()); LY.append(yb.cpu())
    L=torch.cat(LG); Y=torch.cat(LY)
    return total/len(loader.dataset), (L.argmax(1)==Y).float().mean().item()

# Stage 1: freeze warm core, train the rest
model.fc1.weight.requires_grad=False; model.fc1.bias.requires_grad=False
opt1 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=3e-3)
print('Stage 1 (frozen core):')
for e in range(1,13):
    trl,tra = run_epoch(model, train_loader, opt1); val,vaa = run_epoch(model, val_loader)
    if e%3==0 or e==1: print(f'  epoch {e:2d}  train_acc {tra:.3f}  val_acc {vaa:.3f}')

# Stage 2: unfreeze all, fine-tune gently
for p in model.parameters(): p.requires_grad=True
opt2 = torch.optim.Adam(model.parameters(), lr=3e-4)
best,bs,pat,bad = 1e9,None,6,0
print('Stage 2 (fine-tune all):')
for e in range(1,41):
    trl,tra = run_epoch(model, train_loader, opt2); val,vaa = run_epoch(model, val_loader)
    if val < best-1e-4: best,bs,bad = val,deepcopy(model.state_dict()),0; star=' *'
    else: bad+=1; star=''
    if e%4==0 or e==1 or star: print(f'  epoch {e:2d}  train_acc {tra:.3f}  val_acc {vaa:.3f}  val_loss {val:.4f}{star}')
    if bad>=pat: print(f'  early stop at epoch {e}'); break
model.load_state_dict(bs); print('Training done.')


Stage 1 (frozen core):
  epoch  1  train_acc 0.510  val_acc 0.558
  epoch  3  train_acc 0.537  val_acc 0.557
  epoch  6  train_acc 0.542  val_acc 0.558
  epoch  9  train_acc 0.551  val_acc 0.562
  epoch 12  train_acc 0.551  val_acc 0.561
Stage 2 (fine-tune all):
  epoch  1  train_acc 0.550  val_acc 0.562  val_loss 0.9212 *
  epoch  2  train_acc 0.551  val_acc 0.562  val_loss 0.9198 *
  epoch  3  train_acc 0.553  val_acc 0.562  val_loss 0.9188 *
  epoch  4  train_acc 0.551  val_acc 0.562  val_loss 0.9172 *
  epoch  6  train_acc 0.554  val_acc 0.562  val_loss 0.9171 *
  epoch  7  train_acc 0.553  val_acc 0.562  val_loss 0.9170 *
  epoch  8  train_acc 0.553  val_acc 0.562  val_loss 0.9171
  epoch  9  train_acc 0.551  val_acc 0.562  val_loss 0.9154 *
  epoch 12  train_acc 0.553  val_acc 0.561  val_loss 0.9145 *
  epoch 14  train_acc 0.554  val_acc 0.562  val_loss 0.9142 *
  epoch 15  train_acc 0.553  val_acc 0.562  val_loss 0.9140 *
  epoch 16  train_acc 0.553  val_acc 0.562  val_loss 0.91

## Cell 6 — Calibrate (temperature scaling, same as v1)

In [15]:
model.eval()
with torch.no_grad():
    val_logits = model(torch.tensor(Xva).to(DEVICE)).cpu().numpy()
def logloss_at_T(T):
    z=val_logits/T; z=z-z.max(1,keepdims=True); p=np.exp(z); p=p/p.sum(1,keepdims=True)
    return log_loss(yva, np.clip(p,1e-7,1-1e-7), labels=[0,1,2])
Ts=np.linspace(0.5,3.0,60); T_best=float(Ts[int(np.argmin([logloss_at_T(T) for T in Ts]))])
print(f'Best temperature T = {T_best:.3f}  (log-loss {logloss_at_T(1.0):.4f} -> {logloss_at_T(T_best):.4f})')


Best temperature T = 0.839  (log-loss 0.9000 -> 0.8980)


## Cell 7 — Grade on the locked test vault + the v1 vs v2 comparison

In [16]:
def probs_at_T(logits,T):
    z=logits/T; z=z-z.max(1,keepdims=True); p=np.exp(z); return p/p.sum(1,keepdims=True)
with torch.no_grad():
    test_logits = model(torch.tensor(Xte).to(DEVICE)).cpu().numpy()
test_p = probs_at_T(test_logits, T_best); test_pred = test_p.argmax(1)
names=['away','draw','home']

overall_acc = accuracy_score(yte, test_pred)
baseline_acc = max(np.bincount(yte))/len(yte)
cm = confusion_matrix(yte, test_pred, labels=[0,1,2])
draw_recall    = recall_score(yte, test_pred, labels=[1], average='macro', zero_division=0)
draw_precision = precision_score(yte, test_pred, labels=[1], average='macro', zero_division=0)
ll = log_loss(yte, test_p, labels=[0,1,2])

print(f'Overall test accuracy : {overall_acc:.3f}')
print(f'Baseline (always home): {baseline_acc:.3f}')
print(f'Test log-loss         : {ll:.4f}\n')
print('Confusion matrix (rows = truth, cols = guess):')
print('           guess_away  guess_draw  guess_home')
for i,n in enumerate(names):
    print(f'  true_{n}  ' + ''.join(f'{cm[i][j]:11d}' for j in range(3)))
print('\n', classification_report(yte, test_pred, target_names=names, digits=3, zero_division=0))

# ---- v1 numbers (from your run) hard-coded for a fair side-by-side ----
v1 = {'acc':0.469,'logloss':0.9037,'draw_recall':0.825,'draw_precision':0.295,
      'draw_guesses':2974+2889+3927}
v2_draw_guesses = int(cm[:,1].sum())
print('='*56)
print('               v1 (24-12, full wt)   v2 (40-20, sqrt wt)')
print('='*56)
print(f'  accuracy          {v1["acc"]:.3f}                {overall_acc:.3f}')
print(f'  log-loss          {v1["logloss"]:.4f}               {ll:.4f}')
print(f'  draw recall       {v1["draw_recall"]:.3f}                {draw_recall:.3f}')
print(f'  draw precision    {v1["draw_precision"]:.3f}                {draw_precision:.3f}')
print(f'  times shouted draw {v1["draw_guesses"]:>5}                {v2_draw_guesses:>5}')
print('='*56)
print('Reading it: we WANT accuracy up, log-loss down, draw precision up,')
print('and "times shouted draw" down toward the ~3500 real draws in the test set.')


Overall test accuracy : 0.551
Baseline (always home): 0.434
Test log-loss         : 0.9072

Confusion matrix (rows = truth, cols = guess):
           guess_away  guess_draw  guess_home
  true_away         1785        706       2453
  true_draw          282       1126       2093
  true_home          223        942       5302

               precision    recall  f1-score   support

        away      0.779     0.361     0.494      4944
        draw      0.406     0.322     0.359      3501
        home      0.538     0.820     0.650      6467

    accuracy                          0.551     14912
   macro avg      0.575     0.501     0.501     14912
weighted avg      0.587     0.551     0.530     14912

               v1 (24-12, full wt)   v2 (40-20, sqrt wt)
  accuracy          0.469                0.551
  log-loss          0.9037               0.9072
  draw recall       0.825                0.322
  draw precision    0.295                0.406
  times shouted draw  9790                 27

## Cell 8 — Save (only if it actually beat v1) + log to the record

In [17]:
improved = (overall_acc > 0.469) and (draw_precision > 0.295)
tag = 'BETTER than v1' if improved else 'NOT clearly better than v1'
print('Verdict:', tag)

# Always save as v2 (we keep both so we can compare / roll back)
torch.save({'model_state':model.state_dict(),'feature_cols':FEATURE_COLS,
            'arch':{'n_features':N_FEATURES,'n_classes':N_CLASSES,'h1':40,'h2':20},
            'temperature':T_best,'warm_started':bool(TRANSFER_DONE)},
           MODELS/'football_v2.pth')
with open(MODELS/'scaler_v2.pkl','wb') as f: pickle.dump(scaler,f)
print('Saved models/football_v2.pth and models/scaler_v2.pkl')

stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
entry = f'''
## Phase 2b — revised build ({stamp})
- Fixes: sqrt-inverse class weights + bigger net (40->20, {sum(p.numel() for p in model.parameters())} params)
- Test accuracy: {overall_acc:.3f}  (v1 was 0.469, baseline 0.434)
- Test log-loss: {ll:.4f}  (v1 was 0.9037)
- Draw recall: {draw_recall:.3f}  (v1 0.825) | Draw precision: {draw_precision:.3f}  (v1 0.295)
- Times model guessed draw: {v2_draw_guesses} (v1 ~9790, real draws ~3500)
- Verdict: {tag}
- Saved: models/football_v2.pth
'''
rp=RESULTS/'RESULTS.md'
if not rp.exists(): rp.write_text('# World Cup 2026 Win Probability — Results Log\n')
with rp.open('a') as f: f.write(entry)
mc=RESULTS/'metrics.csv'
row=pd.DataFrame([{'timestamp':stamp,'phase':'phase2b_revised','test_acc':round(overall_acc,4),
    'baseline_acc':round(baseline_acc,4),'test_logloss':round(ll,4),
    'draw_recall':round(draw_recall,4),'draw_precision':round(draw_precision,4),
    'temperature':round(T_best,3)}])
row.to_csv(mc, mode='a', header=not mc.exists(), index=False)
print(entry)
print('PHASE 2b COMPLETE')


Verdict: BETTER than v1
Saved models/football_v2.pth and models/scaler_v2.pkl

## Phase 2b — revised build (2026-07-04 09:48 UTC)
- Fixes: sqrt-inverse class weights + bigger net (40->20, 1363 params)
- Test accuracy: 0.551  (v1 was 0.469, baseline 0.434)
- Test log-loss: 0.9072  (v1 was 0.9037)
- Draw recall: 0.322  (v1 0.825) | Draw precision: 0.406  (v1 0.295)
- Times model guessed draw: 2774 (v1 ~9790, real draws ~3500)
- Verdict: BETTER than v1
- Saved: models/football_v2.pth

PHASE 2b COMPLETE
